# W3 Challenge: Customer Sentiment at Scale

**Author:** Alberto Diaz Durana  
**Date:** March 2026  
**Course:** AI in Data Science

## Objective

Repeat the W3 sentiment analysis on the full Customer Support on Twitter dataset (~2.8M tweets) to test how the tools and findings scale. Key questions: do sentiment patterns hold at scale? How does GPU batching affect throughput? What company-level insights emerge from statistically meaningful sample sizes?

## Structure

1. Setup and Data Loading
2. Gemini Exploration (full dataset statistics)
3. AutoViz Visualization (sampled)
4. Sentiment Analysis (GPU-accelerated, 10K sample)
5. Company-Level Analysis
6. Scale Comparison and Reflections

In [ ]:
# Cell 1 - Installs and Imports
import subprocess, sys

packages = [
    'google-genai', 'autoviz', 'transformers', 'torch',
    'pandas', 'matplotlib', 'seaborn'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, re, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google import genai
from transformers import pipeline
import torch
import warnings

print(f"Python: {sys.version}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("All imports successful.")

In [ ]:
# Cell 2 - Data Loading and Initial Inspection
import os

# Local paths (challenge uses full dataset)
local_paths = ['data/twcs/twcs.csv', 'data/twcs.csv']

try:
    import google.colab
    data_dir = '/content/data'
    local_paths = [os.path.join(data_dir, 'twcs.csv')]
except ImportError:
    data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')

df = None
for path in local_paths:
    full_path = path if os.path.isabs(path) else os.path.join(os.path.dirname(os.getcwd()), path)
    if os.path.exists(full_path):
        print(f"Loading from: {full_path}")
        df = pd.read_csv(full_path)
        break

assert df is not None, "Dataset not found. Download: kaggle datasets download -d thoughtvector/customer-support-on-twitter"

print(f"\nShape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nInbound: {df['inbound'].sum():,} ({df['inbound'].mean():.1%})")
print(f"Outbound: {(~df['inbound']).sum():,} ({(~df['inbound']).mean():.1%})")
print(f"\nTop 10 companies (outbound):")
print(df[df['inbound'] == False]['author_id'].value_counts().head(10))
print(f"\nSample rows:")
print(df.head(3).to_string())

In [ ]:
# Cell 3 - Gemini Exploration (Full Dataset)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
except (ImportError, Exception):
    env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.strip().startswith('GOOGLE_API_KEY='):
                    api_key = line.strip().split('=', 1)[1]
if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')

assert api_key, "GOOGLE_API_KEY not found."
client = genai.Client(api_key=api_key)

# Build summary with full dataset stats
inbound = df[df['inbound'] == True]
outbound = df[df['inbound'] == False]
top_companies = outbound['author_id'].value_counts().head(15)

dataset_summary = f"""Dataset: Customer Support on Twitter (FULL)
Shape: {df.shape[0]:,} rows, {df.shape[1]} columns
Columns: {list(df.columns)}

Inbound (customer) tweets: {len(inbound):,} ({len(inbound)/len(df):.1%})
Outbound (company) tweets: {len(outbound):,} ({len(outbound)/len(df):.1%})

Missing values:
{df.isnull().sum().to_string()}

Top 15 companies by response volume:
{top_companies.to_string()}

Sample inbound tweets:
{inbound['text'].sample(5, random_state=42).to_string()}

Sample outbound tweets:
{outbound['text'].sample(5, random_state=42).to_string()}
"""

prompt = f"""You are a data analyst. Given the following dataset summary from the full Customer Support on Twitter corpus (~2.8M tweets), please:
1. Describe the dataset structure and scale
2. Identify patterns in company representation and tweet distribution
3. Suggest analytical directions for sentiment analysis at scale

{dataset_summary}"""

print("Sending to Gemini...\n")
response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
print("=== Gemini's Analysis ===\n")
print(response.text)

In [ ]:
# Cell 4 - AutoViz Visualization (Sampled)
warnings_warn = warnings.warn
warnings.warn = lambda *a, **kw: None

import IPython.core.pylabtools as _pylabtools
_pylabtools.backend2gui = {}

# Sample and add derived features
sample_viz = df.sample(10000, random_state=42).copy()
sample_viz['text_length'] = sample_viz['text'].str.len()
sample_viz['word_count'] = sample_viz['text'].str.split().str.len()

viz_df = sample_viz[['author_id', 'inbound', 'text_length', 'word_count']].copy()
viz_df['inbound'] = viz_df['inbound'].astype(str)

from autoviz import AutoViz_Class
AV = AutoViz_Class()
report = AV.AutoViz(
    filename='', dfte=viz_df, depVar='', verbose=1,
    lowess=False, chart_format='svg',
    max_rows_analyzed=10000, max_cols_analyzed=10
)

warnings.warn = warnings_warn
print(f"\nAutoViz completed on 10K sample.")

In [ ]:
# Cell 5 - Sentiment Analysis (GPU-Accelerated, 10K Sample)

# Filter inbound tweets
inbound_df = df[df['inbound'] == True].copy()
print(f"Total inbound tweets: {len(inbound_df):,}")

# Stratified sample of 10K for sentiment
sample_size = 10_000
sentiment_sample = inbound_df.sample(n=min(sample_size, len(inbound_df)), random_state=42).copy()
print(f"Sentiment sample: {len(sentiment_sample):,} tweets")

# Clean tweets
def clean_tweet(text):
    return re.sub(r'@\S+', '', str(text)).strip()

sentiment_sample['clean_text'] = sentiment_sample['text'].apply(clean_tweet)

# Remove empty texts after cleaning
sentiment_sample = sentiment_sample[sentiment_sample['clean_text'].str.len() > 0].copy()
print(f"After cleaning: {len(sentiment_sample):,} tweets")

# Load model
device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=device,
    batch_size=64
)

# Run with timing
print(f"\nRunning sentiment analysis (device={'GPU' if device == 0 else 'CPU'}, batch_size=64)...")
start = time.time()
results = sentiment_pipe(sentiment_sample['clean_text'].tolist(), truncation=True)
elapsed = time.time() - start

sentiment_sample['sentiment'] = [r['label'] for r in results]
sentiment_sample['confidence'] = [round(r['score'], 3) for r in results]

print(f"Completed in {elapsed:.1f}s ({len(sentiment_sample)/elapsed:.0f} tweets/sec)")
print(f"\n=== Sentiment Distribution ===")
dist = sentiment_sample['sentiment'].value_counts()
for label in ['negative', 'neutral', 'positive']:
    count = dist.get(label, 0)
    pct = count / len(sentiment_sample) * 100
    mean_conf = sentiment_sample[sentiment_sample['sentiment'] == label]['confidence'].mean()
    print(f"  {label:8s}: {count:,} ({pct:.1f}%) | mean confidence: {mean_conf:.3f}")

print(f"\nOverall mean confidence: {sentiment_sample['confidence'].mean():.3f}")

In [ ]:
# Cell 6 - Sentiment Visualization and Company Analysis
os.makedirs("outputs", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
colors = {'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'}
counts = sentiment_sample['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
axes[0].set_title('Sentiment Distribution (10K Inbound Sample)')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 50, f"{val:,}", ha='center', fontweight='bold')

# Confidence by sentiment
for sentiment in ['negative', 'neutral', 'positive']:
    subset = sentiment_sample[sentiment_sample['sentiment'] == sentiment]['confidence']
    axes[1].hist(subset, bins=20, alpha=0.6, label=sentiment, color=colors[sentiment])
axes[1].set_title('Model Confidence by Sentiment')
axes[1].set_xlabel('Confidence Score')
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/challenge_sentiment_dist.png", dpi=150, bbox_inches="tight")
plt.show()

# Company-level sentiment
# Extract company from @mention in tweet text
sentiment_sample['company'] = sentiment_sample['text'].apply(
    lambda t: re.search(r'@(\w+)', str(t)).group(1) if re.search(r'@(\w+)', str(t)) else 'Unknown'
)

company_counts = sentiment_sample['company'].value_counts()
top_companies = company_counts[company_counts >= 50].index.tolist()
print(f"Companies with 50+ tweets: {len(top_companies)}")

top_df = sentiment_sample[sentiment_sample['company'].isin(top_companies)]
cross = pd.crosstab(top_df['company'], top_df['sentiment'], normalize='index') * 100
cross = cross.reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
cross = cross.sort_values('negative', ascending=True)

print(f"\n=== Sentiment by Company (% of tweets, 50+ tweets) ===")
print(cross.round(1).to_string())

# Stacked bar chart
fig, ax = plt.subplots(figsize=(12, max(6, len(cross) * 0.4)))
cross.plot(kind='barh', stacked=True, color=['#e74c3c', '#95a5a6', '#2ecc71'], ax=ax)
ax.set_xlabel('Percentage of Tweets')
ax.set_title('Sentiment by Company (50+ tweets)')
ax.legend(title='Sentiment')
plt.tight_layout()
plt.savefig("outputs/challenge_sentiment_by_company.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 7 - Scale Comparison: 93-Row Sample vs Full Dataset

print("=== W3 (93-row sample) vs Challenge (10K from 2.8M) ===\n")

w3_dist = {'negative': 55, 'neutral': 29, 'positive': 16}
challenge_dist = (sentiment_sample['sentiment'].value_counts(normalize=True) * 100).round(1)

comparison = pd.DataFrame({
    'W3 (93 rows, %)': [w3_dist.get(s, 0) for s in ['negative', 'neutral', 'positive']],
    'Challenge (10K, %)': [challenge_dist.get(s, 0) for s in ['negative', 'neutral', 'positive']]
}, index=['negative', 'neutral', 'positive'])

print(comparison.to_string())
print(f"\nW3 mean confidence: 0.782")
print(f"Challenge mean confidence: {sentiment_sample['confidence'].mean():.3f}")
print(f"\nW3 companies analyzed: 7 (3+ tweets)")
print(f"Challenge companies analyzed: {len(top_companies)} (50+ tweets)")

# Timing comparison
print(f"\nThroughput: {len(sentiment_sample)/elapsed:.0f} tweets/sec")
print(f"Estimated time for full inbound ({len(inbound_df):,} tweets): {len(inbound_df)/max(1, len(sentiment_sample)/elapsed)/60:.1f} min")

## Scale Comparison and Reflections

### What Changes at Scale

Moving from 93 rows to 2.8 million fundamentally changes the analysis in three ways:

1. **Statistical significance.** Company-level sentiment patterns that were anecdotal at 93 rows (3-9 tweets per company) become statistically meaningful. The 10K sample identified 49 companies with 50+ tweets each, compared to 7 companies in the assignment notebook.

2. **Sampling becomes mandatory.** AutoViz would crash on the full dataset; sentiment analysis on 1.5M inbound tweets without GPU batching would take over 8 hours. Every tool required explicit sampling decisions.

3. **Infrastructure matters.** GPU batching at 51 tweets/sec processed the 10K sample in 195 seconds. Estimated time for the full inbound corpus: ~500 minutes. These are real engineering constraints, not academic ones.

### Sentiment at Scale

The 10K sample confirms the W3 finding: customer support tweets skew negative. The distribution shifted slightly, negative dropped from 55% to 51.6%, neutral rose from 29% to 33.4%, positive remained stable at 15-16%. Mean confidence was identical (0.782), suggesting the model's behavior is consistent across sample sizes.

Company-level analysis revealed wide variation: negative sentiment ranged from 19.5% (AirAsiaSupport) to 87.2% among the most complained-about accounts. This granularity, invisible in 93 rows, is the primary analytical gain from scaling up.

### Tool Performance at Scale

- **Gemini:** Scales well because it receives summary statistics, not raw data. The analysis quality depends on the summary, not the dataset size.
- **AutoViz:** Same limitation as W3; text-heavy data gives it little to work with regardless of scale.
- **Hugging Face:** GPU batching transforms feasibility. What would be impractical on CPU becomes routine with batch_size=64 on the Quadro T1000.

### Connection to the Series

The W3 assignment proved the tools work on a curated sample. This challenge tested whether the findings generalize: the negative sentiment skew holds, company patterns persist with greater statistical significance, and GPU acceleration makes the analysis practical at production scale.